# Convert paired FILE_LVL data to CLEAR input

Each source row becomes two adjacent CLEAR records: vulnerable `func_before` (`target=1`) and fixed `func_after` (`target=0`).

In [ ]:
import json
from pathlib import Path

ROOT = Path('/root/CLEAR_Causal_Context_based_Agentic_Reasoning_for_Vulnerability_Detection2')
SOURCE = ROOT / 'dataset/java-pair/java_test_paired_FILE_LVL.json'
OUTPUT = ROOT / 'dataset/java-pair/java_test_paired_CLEAR_FILE_LVL.json'
SOURCE, OUTPUT

In [ ]:
def load_rows(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as fh:
        payload = json.load(fh)
    if isinstance(payload, dict):
        payload = payload.get('data')
    if not isinstance(payload, list) or not all(isinstance(row, dict) for row in payload):
        raise ValueError('Expected a JSON list of objects or an object with a data list')
    return payload

def normalize_cwe(row: dict) -> str:
    value = row.get('cwe_id') or row.get('cwe')
    if not value and row.get('cwe_ids'):
        value = row['cwe_ids'][0]
    return str(value or '').strip()

def convert_to_clear(rows: list[dict]) -> list[dict]:
    converted = []
    for pair_position, row in enumerate(rows):
        before = row.get('func_before')
        after = row.get('func_after')
        if not isinstance(before, str) or not before.strip():
            raise ValueError(f'Row {pair_position}: func_before is empty')
        if not isinstance(after, str) or not after.strip():
            raise ValueError(f'Row {pair_position}: func_after is empty')
        common = {
            'pair_index': row.get('index', pair_position),
            'commit_hash': row.get('commit_hash', ''),
            'file_name': row.get('file_name', ''),
            'cve_id': row.get('cve_id', ''),
            'cwe': normalize_cwe(row),
        }
        converted.append({'index': 2 * pair_position, 'func': before, 'target': 1, **common})
        converted.append({'index': 2 * pair_position + 1, 'func': after, 'target': 0, **common})
    return converted

In [ ]:
source_rows = load_rows(SOURCE)
clear_rows = convert_to_clear(source_rows)

assert len(clear_rows) == 2 * len(source_rows)
assert len({row['index'] for row in clear_rows}) == len(clear_rows)
assert all(clear_rows[i]['target'] == 1 and clear_rows[i + 1]['target'] == 0
           for i in range(0, len(clear_rows), 2))

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT.open('w', encoding='utf-8') as fh:
    json.dump(clear_rows, fh, ensure_ascii=False, indent=2)

print(f'Source pairs: {len(source_rows)}')
print(f'CLEAR records: {len(clear_rows)}')
print(f'Written: {OUTPUT}')

In [ ]:
clear_rows[:2]